# Annotation Quality Control

This notebook audits the existing blinded manual annotation workflow without mutating the source annotation CSV or SQLite database.

It reads the blinded queue through a safe allow-list of columns, reads saved annotations in read-only mode, checks completion and field validity, builds review tables, samples ordinary completed records for QC, and prepares a separate blinded repeat-annotation queue.

The notebook deliberately does not open `master_annotation_sampling_key.csv` and does not load MegaDetector confidence-score or bounding-box columns.

## tl;dr

Run this notebook top-to-bottom. Every generated artifact is written to a fresh timestamped folder under `annotation_quality_control_outputs/`, and every write refuses to overwrite an existing file.

The source files are treated as read-only inputs:

- `sampling_outputs/blinded_manual_annotation_queue.csv`
- `sampling_outputs/manual_annotations.sqlite`
- `sampling_outputs/manual_annotations.csv`

The SQLite database is opened with read-only URI mode. The CSV export is used for reconciliation and as a fallback only if SQLite is unavailable.

## Context & Methods

The manual annotation app stores one row per `annotation_queue_id`. Completed rows must have image quality and the three presence fields populated. Human secondary fields are meaningful only when `manual_human_present = Yes`; vehicle secondary fields are meaningful only when `manual_vehicle_present = Yes`.

This audit checks:

- queue and annotation row counts, missing IDs, duplicate IDs, pending IDs, and annotations not present in the blinded queue;
- invalid enum, score, flag, timestamp, and duration fields;
- inconsistent human and vehicle secondary fields;
- stored `needs_review` flags against the protocol-derived review rule;
- review cases from uncertainty, unusable images, privacy/safeguarding flags, high vehicle privacy risk, and validation issues;
- a random sample of 50 otherwise normal completed rows for QC;
- a separate random 5% blinded repeat-annotation queue with new queue IDs.

In [1]:
from __future__ import annotations

from datetime import datetime, timezone
from pathlib import Path
import hashlib
import json
import math
import sqlite3

import numpy as np
import pandas as pd

BASE_DIR = Path.cwd()
OUTPUT_DIR = BASE_DIR / 'sampling_outputs'
QUEUE_PATH = OUTPUT_DIR / 'blinded_manual_annotation_queue.csv'
DATABASE_PATH = OUTPUT_DIR / 'manual_annotations.sqlite'
ANNOTATION_CSV_PATH = OUTPUT_DIR / 'manual_annotations.csv'
MASTER_KEY_PATH = OUTPUT_DIR / 'master_annotation_sampling_key.csv'
QC_ROOT = BASE_DIR / 'annotation_quality_control_outputs'

SAMPLE_SEED = 20260822
NORMAL_QC_SAMPLE_SIZE = 50
REPEAT_SAMPLE_FRACTION = 0.05
EXPECTED_PROTOCOL_VERSION = '1.1'

BLOCKED_COLUMN_PATTERNS = (
    'confidence',
    'megadetector',
    'mega_detector',
    'detector',
    'bbox',
    'bounding_box',
    'sampling_key',
    'master_key',
    'md_',
)

SAFE_QUEUE_COLUMNS = [
    'annotation_queue_id',
    'annotation_image_url',
    'photo_id',
    'sequence_id',
    'site_id',
    'filename',
    'dirname',
]
REQUIRED_QUEUE_COLUMNS = {'annotation_queue_id', 'annotation_image_url'}

ANNOTATION_COLUMNS = [
    'annotation_queue_id',
    'annotation_status',
    'manual_human_present',
    'manual_animal_present',
    'manual_vehicle_present',
    'annotation_quality',
    'human_recognisability',
    'child_or_vulnerable_person_visible',
    'human_privacy_risk',
    'safeguarding_risk',
    'vehicle_identifiability',
    'vehicle_privacy_risk',
    'notes',
    'protocol_version',
    'annotator_id',
    'annotation_started_at',
    'annotated_at',
    'annotation_duration_seconds',
    'is_repeat_annotation',
    'needs_review',
    'updated_at',
]

PRESENCE_VALUES = {'Yes', 'No', 'Uncertain'}
QUALITY_VALUES = {'Clear', 'Difficult but usable', 'Unusable'}
STATUS_VALUES = {'completed', 'skipped'}
CHILD_VALUES = {'No', 'Yes', 'Uncertain'}
SCORE_VALUES = {0, 1, 2}
FLAG_VALUES = {0, 1}

RUN_STAMP = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
written_outputs: list[str] = []
warnings: list[str] = []


def read_csv_header(path: Path) -> list[str]:
    return [str(column).strip() for column in pd.read_csv(path, nrows=0).columns]


def has_blocked_pattern(column: str) -> bool:
    lowered = str(column).lower()
    return any(pattern in lowered for pattern in BLOCKED_COLUMN_PATTERNS)


def unique_run_dir(root: Path, stem: str) -> Path:
    root.mkdir(parents=True, exist_ok=True)
    candidate = root / stem
    counter = 1
    while candidate.exists():
        candidate = root / f'{stem}_{counter:02d}'
        counter += 1
    candidate.mkdir(parents=False, exist_ok=False)
    return candidate


QC_DIR = unique_run_dir(QC_ROOT, f'qc_run_{RUN_STAMP}')


def relative_output_path(path: Path) -> str:
    return str(path.relative_to(BASE_DIR))


def output_path(filename: str) -> Path:
    path = QC_DIR / filename
    if path.exists():
        raise FileExistsError(f'Refusing to overwrite existing QC output: {path}')
    return path


def write_csv(frame: pd.DataFrame, filename: str) -> Path:
    path = output_path(filename)
    frame.to_csv(path, index=False)
    written_outputs.append(relative_output_path(path))
    return path


def write_json(payload: dict, filename: str) -> Path:
    path = output_path(filename)
    path.write_text(json.dumps(payload, indent=2, default=str), encoding='utf-8')
    written_outputs.append(relative_output_path(path))
    return path


print(f'QC outputs will be written to: {QC_DIR}')

QC outputs will be written to: c:\Users\cheng\Desktop\dissertation1-main\supplementary_var_chunk\annotation_quality_control_outputs\qc_run_20260822T061645Z


### 1. Load The Blinded Queue Safely

The queue is read with an explicit allow-list. If confidence, detector, bounding-box, or key-like columns are present, their names are recorded as blocked, but their values are not loaded.

In [2]:
if not QUEUE_PATH.exists():
    raise FileNotFoundError(f'Blinded queue not found: {QUEUE_PATH}')

queue_header = read_csv_header(QUEUE_PATH)
blocked_queue_columns = [column for column in queue_header if has_blocked_pattern(column)]
missing_queue_columns = sorted(REQUIRED_QUEUE_COLUMNS - set(queue_header))
if missing_queue_columns:
    raise ValueError(f'Queue is missing required columns: {missing_queue_columns}')

safe_queue_column_set = set(SAFE_QUEUE_COLUMNS)
queue = pd.read_csv(
    QUEUE_PATH,
    usecols=lambda column: str(column).strip() in safe_queue_column_set,
    dtype='string',
    low_memory=False,
)
queue.columns = [str(column).strip() for column in queue.columns]

for column in ['annotation_queue_id', 'annotation_image_url']:
    queue[column] = queue[column].astype('string').str.strip()

if blocked_queue_columns:
    warnings.append(
        'Blocked queue columns were detected by name and deliberately not loaded: '
        + ', '.join(blocked_queue_columns)
    )

queue_profile = pd.DataFrame(
    [
        {'metric': 'queue_rows', 'value': len(queue)},
        {'metric': 'queue_columns_loaded', 'value': len(queue.columns)},
        {'metric': 'blocked_queue_columns_not_loaded', 'value': len(blocked_queue_columns)},
        {'metric': 'master_sampling_key_loaded', 'value': False},
    ]
)
display(queue_profile)

,metric,value
0,queue_rows,2077
1,queue_columns_loaded,7
2,blocked_queue_columns_not_loaded,0
3,master_sampling_key_loaded,False


### 2. Load Saved Annotations Read-Only

SQLite is the primary store created by the Streamlit app. This notebook opens it in read-only mode and selects only the annotation table columns used by the app. The CSV export is loaded with the same allow-list so it can be reconciled against SQLite.

In [3]:
def ensure_annotation_columns(frame: pd.DataFrame) -> pd.DataFrame:
    frame = frame.copy()
    frame.columns = [str(column).strip() for column in frame.columns]
    for column in ANNOTATION_COLUMNS:
        if column not in frame.columns:
            frame[column] = pd.NA
    return frame[ANNOTATION_COLUMNS]


def quote_identifier(name: str) -> str:
    quote = chr(34)
    return quote + name.replace(quote, quote + quote) + quote


def load_annotations_from_sqlite(path: Path) -> tuple[pd.DataFrame, str]:
    if not path.exists():
        return pd.DataFrame(columns=ANNOTATION_COLUMNS), 'missing'
    try:
        uri = path.resolve().as_uri() + '?mode=ro'
        with sqlite3.connect(uri, uri=True) as connection:
            table_exists = connection.execute(
                "SELECT 1 FROM sqlite_master WHERE type = 'table' AND name = 'annotations'"
            ).fetchone()
            if not table_exists:
                return pd.DataFrame(columns=ANNOTATION_COLUMNS), 'annotations_table_missing'

            available_columns = [row[1] for row in connection.execute('PRAGMA table_info(annotations)')]
            select_columns = [column for column in ANNOTATION_COLUMNS if column in available_columns]
            if not select_columns:
                return pd.DataFrame(columns=ANNOTATION_COLUMNS), 'annotation_columns_missing'

            select_sql = ', '.join(quote_identifier(column) for column in select_columns)
            frame = pd.read_sql_query(f'SELECT {select_sql} FROM annotations', connection)
        return ensure_annotation_columns(frame), 'loaded_read_only'
    except Exception as error:
        warnings.append(f'SQLite read-only load failed; CSV fallback may be used. Error: {error}')
        return pd.DataFrame(columns=ANNOTATION_COLUMNS), f'load_failed: {error}'


def load_annotations_from_csv(path: Path) -> tuple[pd.DataFrame, str]:
    if not path.exists():
        return pd.DataFrame(columns=ANNOTATION_COLUMNS), 'missing'
    header = read_csv_header(path)
    annotation_column_set = set(ANNOTATION_COLUMNS)
    frame = pd.read_csv(
        path,
        usecols=lambda column: str(column).strip() in annotation_column_set,
        dtype='string',
        low_memory=False,
    )
    loaded_columns = set(str(column).strip() for column in frame.columns)
    ignored_columns = sorted(set(header) - loaded_columns)
    if ignored_columns:
        warnings.append('Annotation CSV had non-schema columns that were ignored: ' + ', '.join(ignored_columns))
    return ensure_annotation_columns(frame), 'loaded'


sqlite_annotations_raw, sqlite_load_status = load_annotations_from_sqlite(DATABASE_PATH)
csv_annotations_raw, csv_load_status = load_annotations_from_csv(ANNOTATION_CSV_PATH)

if len(sqlite_annotations_raw) > 0:
    annotations_raw = sqlite_annotations_raw.copy()
    annotation_source_used = 'sqlite_read_only'
elif len(csv_annotations_raw) > 0:
    annotations_raw = csv_annotations_raw.copy()
    annotation_source_used = 'csv_export_fallback'
else:
    annotations_raw = pd.DataFrame(columns=ANNOTATION_COLUMNS)
    annotation_source_used = 'none_found'


def normalise_for_compare(frame: pd.DataFrame) -> pd.DataFrame:
    frame = ensure_annotation_columns(frame).copy()
    for column in ANNOTATION_COLUMNS:
        frame[column] = frame[column].astype('string').fillna('').str.strip()
    return frame


reconciliation_rows = [
    {'check': 'sqlite_load_status', 'value': sqlite_load_status},
    {'check': 'csv_load_status', 'value': csv_load_status},
    {'check': 'annotation_source_used', 'value': annotation_source_used},
    {'check': 'sqlite_rows', 'value': len(sqlite_annotations_raw)},
    {'check': 'csv_rows', 'value': len(csv_annotations_raw)},
]

if len(sqlite_annotations_raw) > 0 and len(csv_annotations_raw) > 0:
    sqlite_compare = normalise_for_compare(sqlite_annotations_raw).drop_duplicates('annotation_queue_id', keep='last')
    csv_compare = normalise_for_compare(csv_annotations_raw).drop_duplicates('annotation_queue_id', keep='last')
    sqlite_ids = set(sqlite_compare['annotation_queue_id']) - {''}
    csv_ids = set(csv_compare['annotation_queue_id']) - {''}
    reconciliation_rows.extend(
        [
            {'check': 'ids_only_in_sqlite', 'value': len(sqlite_ids - csv_ids)},
            {'check': 'ids_only_in_csv', 'value': len(csv_ids - sqlite_ids)},
        ]
    )
    common_ids = sorted(sqlite_ids & csv_ids)
    sqlite_common = sqlite_compare.set_index('annotation_queue_id').loc[common_ids]
    csv_common = csv_compare.set_index('annotation_queue_id').loc[common_ids]
    for column in [column for column in ANNOTATION_COLUMNS if column != 'annotation_queue_id']:
        mismatch_count = int((sqlite_common[column] != csv_common[column]).sum())
        if mismatch_count:
            reconciliation_rows.append({'check': f'mismatched_column_{column}', 'value': mismatch_count})

source_reconciliation = pd.DataFrame(reconciliation_rows)
write_csv(source_reconciliation, 'csv_sqlite_reconciliation.csv')
display(source_reconciliation)

,check,value
0,sqlite_load_status,loaded_read_only
1,csv_load_status,loaded
2,annotation_source_used,sqlite_read_only
3,sqlite_rows,850
4,csv_rows,850
5,ids_only_in_sqlite,0
6,ids_only_in_csv,0


## Data

The next cells normalise types for validation, join annotations back to the safe queue fields, and write basic completion outputs.

In [4]:
def missing_series(series: pd.Series) -> pd.Series:
    text = series.astype('string')
    return series.isna() | text.isna() | text.str.strip().isin(['', '<NA>', 'nan', 'None'])


def present_series(series: pd.Series) -> pd.Series:
    return ~missing_series(series)


def clean_text_column(series: pd.Series) -> pd.Series:
    cleaned = series.astype('string').str.strip()
    return cleaned.mask(cleaned.isin(['', '<NA>', 'nan', 'None']))


def normalise_annotations(frame: pd.DataFrame) -> pd.DataFrame:
    frame = ensure_annotation_columns(frame).copy()
    text_columns = [
        'annotation_queue_id',
        'annotation_status',
        'manual_human_present',
        'manual_animal_present',
        'manual_vehicle_present',
        'annotation_quality',
        'child_or_vulnerable_person_visible',
        'notes',
        'protocol_version',
        'annotator_id',
        'annotation_started_at',
        'annotated_at',
        'updated_at',
    ]
    integer_columns = [
        'human_recognisability',
        'human_privacy_risk',
        'safeguarding_risk',
        'vehicle_identifiability',
        'vehicle_privacy_risk',
        'is_repeat_annotation',
        'needs_review',
    ]
    for column in text_columns:
        frame[column] = clean_text_column(frame[column])
    for column in integer_columns:
        frame[column] = pd.to_numeric(frame[column], errors='coerce').astype('Int64')
    frame['annotation_duration_seconds'] = pd.to_numeric(
        frame['annotation_duration_seconds'], errors='coerce'
    )
    return frame


annotations = normalise_annotations(annotations_raw)
queue_ids = set(queue['annotation_queue_id'].dropna().astype(str))

annotations_with_queue = annotations.merge(
    queue,
    on='annotation_queue_id',
    how='left',
    indicator='queue_match',
)

data_profile = pd.DataFrame(
    [
        {'metric': 'queue_rows', 'value': len(queue)},
        {'metric': 'annotation_rows_loaded', 'value': len(annotations)},
        {'metric': 'unique_annotation_ids', 'value': annotations['annotation_queue_id'].nunique(dropna=True)},
        {'metric': 'annotation_source_used', 'value': annotation_source_used},
    ]
)
display(data_profile)

,metric,value
0,queue_rows,2077
1,annotation_rows_loaded,850
2,unique_annotation_ids,850
3,annotation_source_used,sqlite_read_only


In [5]:
queue_duplicate_ids = (
    queue.loc[queue['annotation_queue_id'].duplicated(keep=False)]
    .sort_values('annotation_queue_id')
)
annotation_duplicate_ids = (
    annotations.loc[annotations['annotation_queue_id'].duplicated(keep=False)]
    .sort_values('annotation_queue_id')
)

annotation_ids = set(annotations['annotation_queue_id'].dropna().astype(str))
completed_ids = set(
    annotations.loc[annotations['annotation_status'].eq('completed'), 'annotation_queue_id']
    .dropna()
    .astype(str)
)
skipped_ids = set(
    annotations.loc[annotations['annotation_status'].eq('skipped'), 'annotation_queue_id']
    .dropna()
    .astype(str)
)
resolved_ids = completed_ids | skipped_ids

pending_queue = queue.loc[~queue['annotation_queue_id'].astype(str).isin(resolved_ids)].copy()
annotations_not_in_queue = annotations.loc[
    present_series(annotations['annotation_queue_id'])
    & ~annotations['annotation_queue_id'].astype(str).isin(queue_ids)
].copy()
queue_missing_required = queue.loc[
    missing_series(queue['annotation_queue_id']) | missing_series(queue['annotation_image_url'])
].copy()

status_counts = (
    annotations['annotation_status']
    .fillna('<missing>')
    .value_counts(dropna=False)
    .rename_axis('annotation_status')
    .reset_index(name='rows')
)

completion_summary = pd.DataFrame(
    [
        {'metric': 'queue_rows', 'value': len(queue)},
        {'metric': 'annotation_rows', 'value': len(annotations)},
        {'metric': 'completed_unique_queue_ids', 'value': len(completed_ids)},
        {'metric': 'skipped_unique_queue_ids', 'value': len(skipped_ids)},
        {'metric': 'pending_queue_ids', 'value': len(pending_queue)},
        {'metric': 'queue_duplicate_rows', 'value': len(queue_duplicate_ids)},
        {'metric': 'annotation_duplicate_rows', 'value': len(annotation_duplicate_ids)},
        {'metric': 'annotations_not_in_queue', 'value': len(annotations_not_in_queue)},
        {'metric': 'queue_rows_missing_required_fields', 'value': len(queue_missing_required)},
    ]
)

write_csv(completion_summary, 'completion_summary.csv')
write_csv(status_counts, 'annotation_status_counts.csv')
write_csv(queue_duplicate_ids, 'queue_duplicate_ids.csv')
write_csv(annotation_duplicate_ids, 'annotation_duplicate_ids.csv')
write_csv(queue_missing_required, 'queue_rows_missing_required_fields.csv')
write_csv(annotations_not_in_queue, 'annotations_not_in_queue.csv')
write_csv(pending_queue, 'pending_queue_ids.csv')

display(completion_summary)
display(status_counts)

,metric,value
0,queue_rows,2077
1,annotation_rows,850
2,completed_unique_queue_ids,850
3,skipped_unique_queue_ids,0
4,pending_queue_ids,1227
5,queue_duplicate_rows,0
6,annotation_duplicate_rows,0
7,annotations_not_in_queue,0
8,queue_rows_missing_required_fields,0


,annotation_status,rows
0,completed,850


## Results

The validation checks below turn each failed rule into a row-level issue. These issues feed the review table and also exclude rows from the ordinary QC sample.

In [6]:
issue_frames: list[pd.DataFrame] = []


def add_issue(mask: pd.Series, issue_type: str, severity: str, detail: str) -> None:
    mask = pd.Series(mask, index=annotations.index).fillna(False)
    if not bool(mask.any()):
        return
    frame = annotations.loc[mask, ['annotation_queue_id']].copy()
    frame.insert(0, 'annotation_row_index', frame.index)
    frame['issue_type'] = issue_type
    frame['severity'] = severity
    frame['detail'] = detail
    issue_frames.append(frame[['annotation_queue_id', 'annotation_row_index', 'issue_type', 'severity', 'detail']])


def invalid_text_value(column: str, allowed: set[str]) -> pd.Series:
    return present_series(annotations[column]) & ~annotations[column].isin(allowed)


def invalid_integer_value(column: str, allowed: set[int]) -> pd.Series:
    return present_series(annotations[column]) & ~annotations[column].isin(allowed)


completed = annotations['annotation_status'].eq('completed')
skipped = annotations['annotation_status'].eq('skipped')
known_status = annotations['annotation_status'].isin(STATUS_VALUES)
human_yes = annotations['manual_human_present'].eq('Yes')
vehicle_yes = annotations['manual_vehicle_present'].eq('Yes')

add_issue(missing_series(annotations['annotation_queue_id']), 'missing_annotation_queue_id', 'high', 'Annotation row has no queue ID.')
add_issue(
    present_series(annotations['annotation_queue_id']) & ~annotations['annotation_queue_id'].astype(str).isin(queue_ids),
    'annotation_id_not_in_queue',
    'high',
    'Annotation queue ID is not present in the blinded queue.',
)
add_issue(annotations['annotation_queue_id'].duplicated(keep=False), 'duplicate_annotation_queue_id', 'high', 'Duplicate annotation_queue_id rows were found in the selected annotation source.')
add_issue(invalid_text_value('annotation_status', STATUS_VALUES), 'invalid_annotation_status', 'high', 'annotation_status must be completed or skipped.')
add_issue(missing_series(annotations['annotation_status']), 'missing_annotation_status', 'high', 'annotation_status is required.')

for column in ['manual_human_present', 'manual_animal_present', 'manual_vehicle_present']:
    add_issue(completed & missing_series(annotations[column]), f'missing_{column}', 'high', f'{column} is required for completed annotations.')
    add_issue(invalid_text_value(column, PRESENCE_VALUES), f'invalid_{column}', 'high', f'{column} must be Yes, No, or Uncertain.')

add_issue(completed & missing_series(annotations['annotation_quality']), 'missing_annotation_quality', 'high', 'annotation_quality is required for completed annotations.')
add_issue(invalid_text_value('annotation_quality', QUALITY_VALUES), 'invalid_annotation_quality', 'high', 'annotation_quality must be Clear, Difficult but usable, or Unusable.')

unusable = annotations['annotation_quality'].eq('Unusable')
unusable_presence_mismatch = unusable & ~(
    annotations['manual_human_present'].eq('Uncertain')
    & annotations['manual_animal_present'].eq('Uncertain')
    & annotations['manual_vehicle_present'].eq('Uncertain')
)
add_issue(unusable_presence_mismatch, 'unusable_presence_not_all_uncertain', 'high', 'Unusable images must have all three presence labels set to Uncertain.')

for column in ['human_recognisability', 'human_privacy_risk', 'safeguarding_risk', 'vehicle_identifiability', 'vehicle_privacy_risk']:
    add_issue(invalid_integer_value(column, SCORE_VALUES), f'invalid_{column}', 'high', f'{column} must be 0, 1, 2, or blank.')

add_issue(invalid_text_value('child_or_vulnerable_person_visible', CHILD_VALUES), 'invalid_child_or_vulnerable_person_visible', 'high', 'child_or_vulnerable_person_visible must be No, Yes, Uncertain, or blank.')
add_issue(invalid_integer_value('is_repeat_annotation', FLAG_VALUES), 'invalid_is_repeat_annotation', 'medium', 'is_repeat_annotation must be 0, 1, or blank.')
add_issue(invalid_integer_value('needs_review', FLAG_VALUES), 'invalid_needs_review', 'medium', 'needs_review must be 0, 1, or blank.')
add_issue(completed & missing_series(annotations['is_repeat_annotation']), 'missing_is_repeat_annotation', 'medium', 'Completed annotations should record is_repeat_annotation.')
add_issue(completed & missing_series(annotations['needs_review']), 'missing_needs_review', 'medium', 'Completed annotations should record needs_review.')

add_issue(completed & missing_series(annotations['protocol_version']), 'missing_protocol_version', 'medium', 'Completed annotations should record protocol_version.')
add_issue(completed & present_series(annotations['protocol_version']) & annotations['protocol_version'].ne(EXPECTED_PROTOCOL_VERSION), 'unexpected_protocol_version', 'medium', f'Expected protocol_version {EXPECTED_PROTOCOL_VERSION}.')
add_issue(completed & missing_series(annotations['annotator_id']), 'missing_annotator_id', 'medium', 'Completed annotations should record annotator_id.')
add_issue(completed & missing_series(annotations['annotation_started_at']), 'missing_annotation_started_at', 'medium', 'Completed annotations should record annotation_started_at.')
add_issue(completed & missing_series(annotations['annotated_at']), 'missing_annotated_at', 'medium', 'Completed annotations should record annotated_at.')
add_issue(completed & missing_series(annotations['annotation_duration_seconds']), 'missing_annotation_duration_seconds', 'medium', 'Completed annotations should record annotation_duration_seconds.')
add_issue(completed & annotations['annotation_duration_seconds'].lt(0), 'negative_annotation_duration', 'medium', 'annotation_duration_seconds should not be negative.')

human_secondary_fields = [
    'human_recognisability',
    'child_or_vulnerable_person_visible',
    'human_privacy_risk',
    'safeguarding_risk',
]
for column in human_secondary_fields:
    add_issue(completed & human_yes & missing_series(annotations[column]), f'missing_{column}_when_human_yes', 'high', f'{column} is required when manual_human_present is Yes.')
    add_issue(completed & ~human_yes & present_series(annotations[column]), f'{column}_present_when_human_not_yes', 'high', f'{column} should be blank when manual_human_present is not Yes.')

vehicle_secondary_fields = ['vehicle_identifiability', 'vehicle_privacy_risk']
for column in vehicle_secondary_fields:
    add_issue(completed & vehicle_yes & missing_series(annotations[column]), f'missing_{column}_when_vehicle_yes', 'high', f'{column} is required when manual_vehicle_present is Yes.')
    add_issue(completed & ~vehicle_yes & present_series(annotations[column]), f'{column}_present_when_vehicle_not_yes', 'high', f'{column} should be blank when manual_vehicle_present is not Yes.')

add_issue(
    completed & human_yes & annotations['human_recognisability'].isin([1, 2]) & annotations['human_privacy_risk'].eq(0),
    'human_recognisable_but_privacy_zero',
    'medium',
    'Recognisable or potentially recognisable humans usually imply human_privacy_risk >= 1 under the protocol.',
)

add_issue(
    completed
    & human_yes
    & annotations["human_recognisability"].eq(0)
    & annotations["human_privacy_risk"].eq(2),
    "human_privacy_two_but_not_recognisable",
    "medium",
    "Human privacy risk 2 should be reviewed when human recognisability is 0."
)

add_issue(
    completed & vehicle_yes & annotations['vehicle_identifiability'].isin([1, 2]) & annotations['vehicle_privacy_risk'].eq(0),
    'vehicle_identifiable_but_privacy_zero',
    'medium',
    'Potentially or clearly identifiable vehicles usually imply vehicle_privacy_risk >= 1 under the protocol.',
)

any_uncertain_presence = annotations[
    ['manual_human_present', 'manual_animal_present', 'manual_vehicle_present']
].eq('Uncertain').any(axis=1)
computed_needs_review = completed & (
    any_uncertain_presence
    | annotations['annotation_quality'].eq('Unusable')
    | annotations['human_recognisability'].isin([1, 2])
    | annotations['child_or_vulnerable_person_visible'].isin(['Yes', 'Uncertain'])
    | annotations['human_privacy_risk'].isin([1, 2])
    | annotations['safeguarding_risk'].isin([1, 2])
    | annotations['vehicle_privacy_risk'].eq(2)
)

stored_needs_review = annotations['needs_review'].fillna(0).astype('Int64').eq(1)
add_issue(
    completed & present_series(annotations['needs_review']) & stored_needs_review.ne(computed_needs_review),
    'needs_review_mismatch',
    'medium',
    'Stored needs_review does not match the protocol-derived review flag.',
)

field_issues = (
    pd.concat(issue_frames, ignore_index=True)
    if issue_frames
    else pd.DataFrame(columns=['annotation_queue_id', 'annotation_row_index', 'issue_type', 'severity', 'detail'])
)

issue_summary = (
    field_issues.groupby(['severity', 'issue_type'], dropna=False)
    .size()
    .reset_index(name='rows')
    .sort_values(['severity', 'rows', 'issue_type'], ascending=[True, False, True])
)

write_csv(field_issues, 'annotation_field_issues.csv')
write_csv(issue_summary, 'annotation_field_issue_summary.csv')
display(issue_summary)

,severity,issue_type,rows
1,medium,vehicle_identifiable_but_privacy_zero,33
0,medium,human_recognisable_but_privacy_zero,10


### 3. Summarise Review Cases

Review cases include uncertain presence labels, unusable images, the stored or recomputed review flag, and row-level validation or consistency issues.

In [7]:
review_reason_frames: list[pd.DataFrame] = []


def add_review_reason(mask: pd.Series, reason: str) -> None:
    mask = pd.Series(mask, index=annotations.index).fillna(False)
    if not bool(mask.any()):
        return
    frame = annotations.loc[mask, ['annotation_queue_id']].copy()
    frame.insert(0, 'annotation_row_index', frame.index)
    frame['review_reason'] = reason
    review_reason_frames.append(frame)


add_review_reason(annotations['manual_human_present'].eq('Uncertain'), 'human_presence_uncertain')
add_review_reason(annotations['manual_animal_present'].eq('Uncertain'), 'animal_presence_uncertain')
add_review_reason(annotations['manual_vehicle_present'].eq('Uncertain'), 'vehicle_presence_uncertain')
add_review_reason(annotations['annotation_quality'].eq('Unusable'), 'image_quality_unusable')
add_review_reason(stored_needs_review, 'stored_needs_review')
add_review_reason(computed_needs_review, 'protocol_computed_needs_review')
has_note = (
    annotations["notes"]
    .fillna("")
    .astype(str)
    .str.strip()
    .ne("")
)

add_review_reason(
    has_note,
    "annotation_has_note"
)

if not field_issues.empty:
    issue_reasons = field_issues[['annotation_queue_id', 'annotation_row_index', 'issue_type']].copy()
    issue_reasons = issue_reasons.rename(columns={'issue_type': 'review_reason'})
    issue_reasons['review_reason'] = 'field_issue:' + issue_reasons['review_reason'].astype(str)
    review_reason_frames.append(issue_reasons)

review_reasons = (
    pd.concat(review_reason_frames, ignore_index=True).drop_duplicates()
    if review_reason_frames
    else pd.DataFrame(columns=['annotation_row_index', 'annotation_queue_id', 'review_reason'])
)

review_reason_summary = (
    review_reasons['review_reason']
    .value_counts(dropna=False)
    .rename_axis('review_reason')
    .reset_index(name='rows')
)

review_case_summary = pd.DataFrame(
    [
        {'metric': 'completed_rows', 'value': int(completed.sum())},
        {'metric': 'any_uncertain_presence_rows', 'value': int(any_uncertain_presence.sum())},
        {'metric': 'human_presence_uncertain_rows', 'value': int(annotations['manual_human_present'].eq('Uncertain').sum())},
        {'metric': 'animal_presence_uncertain_rows', 'value': int(annotations['manual_animal_present'].eq('Uncertain').sum())},
        {'metric': 'vehicle_presence_uncertain_rows', 'value': int(annotations['manual_vehicle_present'].eq('Uncertain').sum())},
        {'metric': 'unusable_quality_rows', 'value': int(annotations['annotation_quality'].eq('Unusable').sum())},
        {'metric': 'stored_needs_review_rows', 'value': int(stored_needs_review.sum())},
        {'metric': 'protocol_computed_needs_review_rows', 'value': int(computed_needs_review.sum())},
        {'metric': 'rows_with_field_issues', 'value': int(field_issues['annotation_queue_id'].nunique(dropna=True))},
    ]
)

review_by_row = (
    review_reasons.groupby('annotation_row_index', dropna=False)['review_reason']
    .agg(lambda values: '; '.join(sorted(set(map(str, values)))))
    .reset_index(name='review_reasons')
)

review_table = annotations_with_queue.copy()
review_table.insert(0, 'annotation_row_index', review_table.index)
review_table = review_table.merge(review_by_row, on='annotation_row_index', how='inner')

review_columns = [
    'annotation_queue_id',
    'annotation_image_url',
    'photo_id',
    'sequence_id',
    'site_id',
    'filename',
    'annotation_status',
    'annotation_quality',
    'manual_human_present',
    'manual_animal_present',
    'manual_vehicle_present',
    'human_recognisability',
    'child_or_vulnerable_person_visible',
    'human_privacy_risk',
    'safeguarding_risk',
    'vehicle_identifiability',
    'vehicle_privacy_risk',
    'needs_review',
    'review_reasons',
    'notes',
    'annotated_at',
]
review_columns = [column for column in review_columns if column in review_table.columns]
review_table = review_table[review_columns].sort_values(['annotation_queue_id'], kind='stable')

write_csv(review_case_summary, 'review_case_summary.csv')
write_csv(review_reason_summary, 'review_reason_summary.csv')
write_csv(review_table, 'review_table.csv')

display(review_case_summary)
display(review_reason_summary.head(25))
display(review_table.head(25))

,metric,value
0,completed_rows,850
1,any_uncertain_presence_rows,102
2,human_presence_uncertain_rows,41
3,animal_presence_uncertain_rows,87
4,vehicle_presence_uncertain_rows,43
5,unusable_quality_rows,31
6,stored_needs_review_rows,158
7,protocol_computed_needs_review_rows,158
8,rows_with_field_issues,39


,review_reason,rows
0,stored_needs_review,158
1,protocol_computed_needs_review,158
2,animal_presence_uncertain,87
3,annotation_has_note,60
4,vehicle_presence_uncertain,43
5,human_presence_uncertain,41
6,field_issue:vehicle_identifiable_but_privacy_zero,33
7,image_quality_unusable,31
8,field_issue:human_recognisable_but_privacy_zero,10


,annotation_queue_id,annotation_image_url,photo_id,sequence_id,site_id,filename,annotation_status,annotation_quality,manual_human_present,manual_animal_present,...,human_recognisability,child_or_vulnerable_person_visible,human_privacy_risk,safeguarding_risk,vehicle_identifiability,vehicle_privacy_risk,needs_review,review_reasons,notes,annotated_at
0,ANNOT_0004,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,7467685,29427474,4915.0,7ad92665fb47c8b538e64018a9e9f9b7.jpg,completed,Difficult but usable,No,Uncertain,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,animal_presence_uncertain; protocol_computed_n...,<NA>,2026-07-29T13:59:14+00:00
1,ANNOT_0008,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,10191095,30020607,5432.0,8a84ad21bb7400d39c323afff74001b1.jpg,completed,Clear,Yes,No,...,0,No,1,0,<NA>,<NA>,1,annotation_has_note; protocol_computed_needs_r...,Person partially obscured by hedge at right ed...,2026-07-29T14:05:22+00:00
2,ANNOT_0009,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,15773603,31344087,6357.0,8c9c41937c21c91497f7997ed2176b45.jpg,completed,Difficult but usable,No,Uncertain,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,animal_presence_uncertain; annotation_has_note...,Blurred central shape may be part of an animal...,2026-07-29T14:07:12+00:00
3,ANNOT_0013,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,18403529,31896482,6977.0,580e423f77bc72225c9e75b22cd8ee8f.jpg,completed,Clear,No,No,...,<NA>,<NA>,<NA>,<NA>,2,0,0,field_issue:vehicle_identifiable_but_privacy_zero,<NA>,2026-08-05T02:46:40+00:00
4,ANNOT_0019,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,13845314,30918906,6177.0,00d5dd7b828a0cf2bcb7f07c0c52d28a.jpg,completed,Clear,No,Uncertain,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,animal_presence_uncertain; protocol_computed_n...,<NA>,2026-08-05T02:50:43+00:00
5,ANNOT_0026,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,16500592,31494596,6662.0,a018350cb7611e58391a1de5137ed505.jpg,completed,Clear,No,No,...,<NA>,<NA>,<NA>,<NA>,1,0,0,field_issue:vehicle_identifiable_but_privacy_zero,<NA>,2026-08-05T02:55:19+00:00
6,ANNOT_0027,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,10997381,30224718,5552.0,1624ff157553a3f4596af6cad044121e.jpg,completed,Clear,Yes,No,...,2,No,1,0,<NA>,<NA>,1,protocol_computed_needs_review; stored_needs_r...,<NA>,2026-08-05T03:01:22+00:00
7,ANNOT_0028,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,10059218,29982748,5417.0,0881074c9e73af1f0180e46b7469095d.jpg,completed,Clear,Yes,No,...,1,No,0,0,<NA>,<NA>,1,field_issue:human_recognisable_but_privacy_zer...,<NA>,2026-08-05T03:02:00+00:00
8,ANNOT_0032,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,12216290,30507588,4859.0,e149f6c84e897d81d1751692ddddea8b.jpg,completed,Difficult but usable,No,Yes,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,0,annotation_has_note,only eyes are visible,2026-08-05T03:04:09+00:00
9,ANNOT_0037,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,8338420,29586350,5071.0,75e1d91621349868dee5d37fa1279083.jpg,completed,Difficult but usable,No,Uncertain,...,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,1,animal_presence_uncertain; protocol_computed_n...,<NA>,2026-08-05T03:05:59+00:00


### 4. Randomly Sample Otherwise Normal Completed Records

This sample is for manual QC of ordinary completed annotations. It excludes rows with any review reason, invalid field issue, duplicate/missing ID issue, `Unusable` quality, `Uncertain` presence, repeat annotations, and records missing from the queue.

In [8]:
issue_row_indices = set(field_issues['annotation_row_index'].dropna().astype(int)) if not field_issues.empty else set()
review_row_indices = set(review_reasons['annotation_row_index'].dropna().astype(int)) if not review_reasons.empty else set()

normal_candidates = annotations_with_queue.copy()
normal_candidates.insert(0, 'annotation_row_index', normal_candidates.index)

normal_mask = (
    normal_candidates['annotation_status'].eq('completed')
    & normal_candidates['queue_match'].eq('both')
    & normal_candidates['is_repeat_annotation'].fillna(0).astype('Int64').ne(1)
    & normal_candidates['annotation_quality'].ne('Unusable')
    & ~normal_candidates[['manual_human_present', 'manual_animal_present', 'manual_vehicle_present']].eq('Uncertain').any(axis=1)
    & normal_candidates['needs_review'].fillna(0).astype('Int64').ne(1)
    & ~normal_candidates['annotation_row_index'].isin(issue_row_indices)
    & ~normal_candidates['annotation_row_index'].isin(review_row_indices)
)

normal_sample_base = normal_candidates.loc[normal_mask].drop_duplicates('annotation_queue_id', keep='last')
normal_sample_size = min(NORMAL_QC_SAMPLE_SIZE, len(normal_sample_base))
if normal_sample_size < NORMAL_QC_SAMPLE_SIZE:
    warnings.append(
        f'Only {normal_sample_size} otherwise normal completed records were available for the requested {NORMAL_QC_SAMPLE_SIZE}-row QC sample.'
    )

normal_qc_sample = (
    normal_sample_base.sample(n=normal_sample_size, random_state=SAMPLE_SEED)
    if normal_sample_size > 0
    else normal_sample_base.copy()
)

normal_sample_columns = [
    'annotation_queue_id',
    'annotation_image_url',
    'photo_id',
    'sequence_id',
    'site_id',
    'filename',
    'annotation_quality',
    'manual_human_present',
    'manual_animal_present',
    'manual_vehicle_present',
    'human_recognisability',
    'child_or_vulnerable_person_visible',
    'human_privacy_risk',
    'safeguarding_risk',
    'vehicle_identifiability',
    'vehicle_privacy_risk',
    'annotated_at',
    'notes',
]
normal_sample_columns = [column for column in normal_sample_columns if column in normal_qc_sample.columns]
normal_qc_sample = normal_qc_sample[normal_sample_columns].sort_values('annotation_queue_id', kind='stable')

write_csv(normal_qc_sample, 'normal_completed_qc_sample_50.csv')
display(pd.DataFrame([{'normal_candidate_rows': len(normal_sample_base), 'sampled_rows': len(normal_qc_sample)}]))
display(normal_qc_sample.head(20))

,normal_candidate_rows,sampled_rows
0,636,50


,annotation_queue_id,annotation_image_url,photo_id,sequence_id,site_id,filename,annotation_quality,manual_human_present,manual_animal_present,manual_vehicle_present,human_recognisability,child_or_vulnerable_person_visible,human_privacy_risk,safeguarding_risk,vehicle_identifiability,vehicle_privacy_risk,annotated_at,notes
16,ANNOT_0017,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,26048699,33657402,8360.0,317c22042929cbf47c73d3cb8a3fd0f4_r.jpg,Difficult but usable,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T02:49:59+00:00,<NA>
20,ANNOT_0021,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,17796707,31773965,6867.0,03b1411d8f955e78922e8935ed304da5.jpg,Clear,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T02:51:37+00:00,<NA>
42,ANNOT_0043,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,18812093,31998967,5613.0,b0339785bbb2686dce4f5b113cc0591d.jpg,Clear,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T03:10:57+00:00,<NA>
54,ANNOT_0055,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,23058557,32926378,7774.0,c206e4edfc4d5c63097fffdaa05125eb.jpg,Clear,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T03:48:32+00:00,<NA>
60,ANNOT_0061,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,17864257,31789195,6720.0,c8d064f47c4af5604f6a7d441396c4b5.jpg,Clear,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T03:52:28+00:00,<NA>
139,ANNOT_0140,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,16974993,31609504,6706.0,561da1ee880b917124ffe9a52a83c10a.jpg,Clear,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T07:41:54+00:00,<NA>
173,ANNOT_0174,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,26485990,33768492,7874.0,f1e19e85fc1bcf6b7eac05e47e3a8034_r.jpg,Clear,No,No,Yes,<NA>,<NA>,<NA>,<NA>,2,1,2026-08-05T08:07:26+00:00,<NA>
174,ANNOT_0175,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,12564479,30582928,5844.0,546302ef69c75f1121f66e92abcc8060.jpg,Difficult but usable,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T08:07:58+00:00,<NA>
195,ANNOT_0196,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,13586393,30844887,6100.0,bc79e718fb17197d70a7493d9488d069.jpg,Clear,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T08:17:11+00:00,<NA>
208,ANNOT_0209,https://mammalweb.s3-eu-west-1.amazonaws.com/p...,13861479,30923973,6185.0,d44aeb4c2f1c3b5b924f334500e54b70.jpg,Difficult but usable,No,No,No,<NA>,<NA>,<NA>,<NA>,<NA>,<NA>,2026-08-05T08:25:24+00:00,<NA>


### 5. Prepare A Separate 5% Blinded Repeat-Annotation Sample

The repeat sample is drawn from completed non-repeat annotations that are present in the blinded queue. The output queue contains only new repeat queue IDs and image URLs so it can be used for a later blinded session.

A separate internal key is saved in the QC folder so agreement can be calculated after repeat annotation. Do not give that key to the annotator during the repeat session.

In [9]:
repeat_candidates = annotations_with_queue.copy()
repeat_candidates.insert(0, 'annotation_row_index', repeat_candidates.index)
repeat_mask = (
    repeat_candidates['annotation_status'].eq('completed')
    & repeat_candidates['queue_match'].eq('both')
    & repeat_candidates['is_repeat_annotation'].fillna(0).astype('Int64').ne(1)
    & present_series(repeat_candidates['annotation_image_url'])
    & present_series(repeat_candidates['annotation_queue_id'])
)
repeat_sample_base = repeat_candidates.loc[repeat_mask].drop_duplicates('annotation_queue_id', keep='last')
repeat_sample_size = int(math.ceil(len(repeat_sample_base) * REPEAT_SAMPLE_FRACTION)) if len(repeat_sample_base) else 0

repeat_selected = (
    repeat_sample_base.sample(n=repeat_sample_size, random_state=SAMPLE_SEED + 1)
    if repeat_sample_size > 0
    else repeat_sample_base.copy()
)


def repeat_queue_id(original_queue_id: str) -> str:
    digest = hashlib.sha256(f'{SAMPLE_SEED}:{original_queue_id}'.encode('utf-8')).hexdigest()[:12]
    return f'repeat_{digest}'


repeat_key = repeat_selected[['annotation_queue_id', 'annotation_image_url']].copy()
repeat_key = repeat_key.rename(columns={'annotation_queue_id': 'original_annotation_queue_id'})
repeat_key.insert(
    0,
    'repeat_annotation_queue_id',
    repeat_key['original_annotation_queue_id'].astype(str).map(repeat_queue_id),
)

if repeat_key['repeat_annotation_queue_id'].duplicated().any():
    raise ValueError('Repeat queue ID collision detected; change SAMPLE_SEED and rerun in a new QC folder.')

blinded_repeat_queue = repeat_key[['repeat_annotation_queue_id', 'annotation_image_url']].rename(
    columns={'repeat_annotation_queue_id': 'annotation_queue_id'}
)

write_csv(blinded_repeat_queue, 'blinded_repeat_annotation_queue_5pct.csv')
write_csv(repeat_key, 'repeat_annotation_internal_key_5pct.csv')

repeat_summary = pd.DataFrame(
    [
        {'metric': 'repeat_candidate_rows', 'value': len(repeat_sample_base)},
        {'metric': 'repeat_fraction', 'value': REPEAT_SAMPLE_FRACTION},
        {'metric': 'repeat_sample_rows', 'value': len(blinded_repeat_queue)},
    ]
)
write_csv(repeat_summary, 'repeat_annotation_sample_summary.csv')

display(repeat_summary)
display(blinded_repeat_queue.head(20))

,metric,value
0,repeat_candidate_rows,850.00
1,repeat_fraction,0.05
2,repeat_sample_rows,43.00


,annotation_queue_id,annotation_image_url
364,repeat_4ca187671fe2,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
716,repeat_fe5af562f92f,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
296,repeat_5c4ade0fc47f,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
84,repeat_93201fa91810,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
62,repeat_ec8242a49d9b,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
185,repeat_34ccdf3c60bc,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
164,repeat_658298b3b9b3,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
433,repeat_3db40064133f,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
637,repeat_1210adfae669,https://mammalweb.s3-eu-west-1.amazonaws.com/p...
394,repeat_3d4b6d86bef6,https://mammalweb.s3-eu-west-1.amazonaws.com/p...


## Takeaways

The final cell writes a run manifest and output index. Review `completion_summary.csv`, `annotation_field_issue_summary.csv`, `review_case_summary.csv`, and `review_table.csv` first. Then use `normal_completed_qc_sample_50.csv` for ordinary QC and `blinded_repeat_annotation_queue_5pct.csv` for the later blinded repeat session.

In [10]:
manifest = {
    'created_at_utc': RUN_STAMP,
    'base_dir': str(BASE_DIR),
    'qc_dir': str(QC_DIR),
    'source_paths': {
        'queue': str(QUEUE_PATH),
        'sqlite_annotations': str(DATABASE_PATH),
        'csv_annotations': str(ANNOTATION_CSV_PATH),
    },
    'annotation_source_used': annotation_source_used,
    'master_sampling_key_path_known_but_not_loaded': str(MASTER_KEY_PATH),
    'master_sampling_key_loaded': False,
    'blocked_queue_columns_detected_but_not_loaded': blocked_queue_columns,
    'parameters': {
        'sample_seed': SAMPLE_SEED,
        'normal_qc_sample_size_requested': NORMAL_QC_SAMPLE_SIZE,
        'repeat_sample_fraction': REPEAT_SAMPLE_FRACTION,
        'expected_protocol_version': EXPECTED_PROTOCOL_VERSION,
    },
    'row_counts': {
        'queue_rows': len(queue),
        'annotation_rows': len(annotations),
        'completed_unique_queue_ids': len(completed_ids),
        'skipped_unique_queue_ids': len(skipped_ids),
        'pending_queue_ids': len(pending_queue),
        'review_table_rows': len(review_table),
        'normal_qc_sample_rows': len(normal_qc_sample),
        'repeat_sample_rows': len(blinded_repeat_queue),
    },
    'warnings': warnings,
    'outputs_written_before_manifest': written_outputs.copy(),
}

write_json(manifest, 'qc_run_manifest.json')
output_index = pd.DataFrame({'output_path': written_outputs})
write_csv(output_index, 'qc_output_index.csv')

print(f'QC run complete. Outputs: {QC_DIR}')
display(output_index)

QC run complete. Outputs: c:\Users\cheng\Desktop\dissertation1-main\supplementary_var_chunk\annotation_quality_control_outputs\qc_run_20260822T061645Z


,output_path
0,annotation_quality_control_outputs\qc_run_2026...
1,annotation_quality_control_outputs\qc_run_2026...
2,annotation_quality_control_outputs\qc_run_2026...
3,annotation_quality_control_outputs\qc_run_2026...
4,annotation_quality_control_outputs\qc_run_2026...
5,annotation_quality_control_outputs\qc_run_2026...
6,annotation_quality_control_outputs\qc_run_2026...
7,annotation_quality_control_outputs\qc_run_2026...
8,annotation_quality_control_outputs\qc_run_2026...
9,annotation_quality_control_outputs\qc_run_2026...
